In [4]:
import os
import sys
import tempfile
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# 1) Basis-Verzeichnisse definieren
repo_root = Path.cwd()
scripts_dir = repo_root / "scripts"
src_root    = repo_root / "src"

# 2) Temp-Ordner anlegen und Paket kopieren
tmp_dir = Path(tempfile.mkdtemp())
if (src_root / "FIT_python").exists():
    pkg_src = src_root / "FIT_python"
elif (repo_root / "FIT_python").exists():
    pkg_src = repo_root / "FIT_python"
else:
    raise FileNotFoundError("Kein FIT_python-Ordner gefunden.")
pkg_dst = tmp_dir / "FIT_python"
shutil.copytree(pkg_src, pkg_dst)

# 3) Minimal config.py erstellen, falls nicht vorhanden
cfg = pkg_dst / "config.py"
if not cfg.exists():
    cfg.write_text("""from pathlib import Path
PROJECT_ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
CLEANED_DIR = DATA_DIR / 'cleaned'
SPLITS_DIR = DATA_DIR / 'processed' / 'splits'
SCALED_DIR = DATA_DIR / 'processed' / 'scaled'
FEATURE_SELECTED_DIR = DATA_DIR / 'processed' / 'feature_selected'
DIM_REDUCED_DIR = DATA_DIR / 'processed' / 'dim_reduced'
RESULTS_DATA_DIR = PROJECT_ROOT / 'results' / 'data'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
GLOBAL_RANDOM_SEED = 42
NUM_FOLDS = 3
""")

# 4) Dummy-Rohdaten erzeugen
raw_dir = tmp_dir / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
df = pd.DataFrame({
    "species": ["otter"]*8,
    "individual_id": [1,1,2,2,3,3,4,4],
    "trail": [1,2,1,2,1,2,1,2],
    "sex": ["F","F","M","M","F","M","F","M"],
    "dataorigin": ["OwnData"]*6 + ["Field"]*2,
    "feat1": range(8), "feat2": range(10,18), "feat3": np.linspace(0,1,8),
})
df.to_csv(raw_dir/"demo_otter.csv", index=False)

# 5) PYTHONPATH setzen
os.environ["PYTHONPATH"] = f"{tmp_dir}{os.pathsep}{src_root}"

# Schritt 1: Cleaning
subprocess.run([sys.executable, str(scripts_dir/"clean_data.py"),
                "--raw-dir", str(tmp_dir/"data"/"raw"),
                "--cleaned-dir", str(tmp_dir/"data"/"cleaned")],
               check=True)
# Prüfung Cleaning
cleaned = list((tmp_dir/"data"/"cleaned").glob("*.parquet"))[0]
print("Clean head:"); display(pd.read_parquet(cleaned).head())

# Schritt 2: Splitting
subprocess.run([sys.executable, str(scripts_dir/"create_splits.py"),
                "--cleaned-dir", str(tmp_dir/"data"/"cleaned"),
                "--splits-dir", str(tmp_dir/"data"/"processed"/"splits")],
               check=True)
# Prüfung Splitting
train_file = tmp_dir/"data"/"processed"/"splits"/"demo_otter_train.parquet"
print("Split head:"); display(pd.read_parquet(train_file).head())

# Schritt 3: Summary
subprocess.run([sys.executable, str(scripts_dir/"summary.py"),
                "--splits-dir", str(tmp_dir/"data"/"processed"/"splits"),
                "--output-table", str(tmp_dir/"results"/"data"/"summary.csv"),
                "--fig-dir", str(tmp_dir/"results"/"figures"),
                "--no-plot"],
               check=True)
# Prüfung Summary
print("Summary head:"); display(pd.read_csv(tmp_dir/"results"/"data"/"summary.csv").head())

# Schritt 4: Scaling
subprocess.run([sys.executable, str(scripts_dir/"scale_splits.py"),
                "--splits-dir", str(tmp_dir/"data"/"processed"/"splits"),
                "--scaled-dir", str(tmp_dir/"data"/"processed"/"scaled")],
               check=True)
# Prüfung Scaling
scaled = tmp_dir/"data"/"processed"/"scaled"/"demo_otter_train.parquet"
df_scaled = pd.read_parquet(scaled)
print("Scaled mean/std:"); print(df_scaled[["feat1","feat2","feat3"]].agg(["mean","std"]))

# Schritt 5: Feature Selection
subprocess.run([sys.executable, str(scripts_dir/"create_feature_selection.py"),
                "--scaled-dir", str(tmp_dir/"data"/"processed"/"scaled"),
                "--feature-dir", str(tmp_dir/"data"/"processed"/"feature_selected")],
               check=True)
# Prüfung Feature Selection
fs = tmp_dir/"data"/"processed"/"feature_selected"/"demo_otter_train.parquet"
print("Feature cols:"); print(pd.read_parquet(fs).columns.tolist())

# Schritt 6: Dimensionality Reduction
subprocess.run([sys.executable, str(scripts_dir/"create_dim_reduction.py"),
                "--feature-dir", str(tmp_dir/"data"/"processed"/"feature_selected"),
                "--dim-dir", str(tmp_dir/"data"/"processed"/"dim_reduced")],
               check=True)
# Prüfung Dim Reduction
dr = tmp_dir/"data"/"processed"/"dim_reduced"/"demo_otter_train.parquet"
print("Dim cols:"); print([c for c in pd.read_parquet(dr).columns if c.startswith("PC")])

# Schritt 7: Model Comparison
subprocess.run([sys.executable, str(scripts_dir/"compare_models.py"),
                "--dim-dir", str(tmp_dir/"data"/"processed"/"dim_reduced"),
                "--results-dir", str(tmp_dir/"results"/"data")],
               check=True)
# Prüfung Model Comparison
mc = tmp_dir/"results"/"data"/"model_comparison.csv"
print("Model comparison head:"); display(pd.read_csv(mc).head())

print("## Integrationstest abgeschlossen!")


FileNotFoundError: Kein FIT_python-Ordner gefunden.